# Stage 5 — explainability and efficiency

**Runtime → Change runtime type → T4 GPU → Save. Then Runtime → Run all.**

About **40 minutes**, mostly unattended.

---

### What this stage is for

The rest of the project has shown that *removing* the RR timing features destroys
Class S performance. That is evidence about the feature set. It is not evidence
that the trained model actually **uses** them — a model can be helped by a
feature at training time and lean on something else at prediction time.

This stage asks the model directly. It blanks out one region of the heartbeat at
a time — the P wave, the QRS complex, the T wave, or the four timing features —
and measures how much the model's confidence in the right answer falls. A big
fall means it was relying on that region.

Then it checks whether that story is trustworthy, by deleting regions in the
order the attribution claims matter most and seeing whether the model degrades
faster than if the regions were deleted at random. If it does not, the
attribution is decoration and we say so.

Finally it measures parameters, model size and per-beat latency.

### Two things to know before you start

**DS2 is never opened.** This stage works entirely inside DS1, and on beats each
model did not train on. Nothing here can affect the Stage 4 result.

**You can run it as many times as you like.** Unlike Stage 4, there is no
one-shot rule here. If something looks wrong, re-run it.

## 1. Confirm a GPU was allocated

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

print('GPU   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

## 2. Get the code

Public repository — nothing to request, nothing to upload.

In [ ]:
%cd /content
!rm -rf ecg-paper
!git clone --depth 1 https://github.com/Rishabhhcodes/ecg-paper.git
%cd /content/ecg-paper

!pip install -q wfdb xgboost imbalanced-learn matplotlib

import subprocess
print('\ncommit :', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                                    capture_output=True, text=True).stdout.strip())

import os
assert os.path.exists('src/stage5_explain.py'), (
    'src/stage5_explain.py is not in the repository yet. Tell Rishabh — the '
    'notebook clones main, so the script has to be merged first.')
print('stage5_explain.py present')

## 3. The dataset

You need `artifacts/ds1.npz`. There are two ways to get it; **use whichever
Rishabh told you**.

**A — he sent you the file.** Run the cell below and upload `ds1.npz`
(about 12 MB). This is the faster path and it uses the exact dataset every
earlier stage used.

**B — build it yourself.** Skip the upload cell and run the build cell instead.
It downloads ~90 MB from PhysioNet, which is sometimes slow and sometimes drops
the connection; the script retries on its own.

In [ ]:
# --- A: upload ds1.npz -------------------------------------------------
import glob, os, shutil
import numpy as np
from google.colab import files

os.makedirs('artifacts', exist_ok=True)
files.upload()                      # pick ds1.npz

# An .npz IS a zip file. Move it; never unpack it, or the arrays scatter
# into the working directory and the file itself disappears.
for name in glob.glob('*.npz'):
    shutil.move(name, os.path.join('artifacts', os.path.basename(name)))

p = 'artifacts/ds1.npz'
assert os.path.exists(p), 'ds1.npz missing — re-run this cell and pick the file'
d = np.load(p)
assert sorted(d.files) == ['X', 'pid', 'rr', 'y'], sorted(d.files)
assert len(d['y']) == 50977, f"{len(d['y']):,} beats, expected 50,977"
print(f"ds1.npz  {os.path.getsize(p)/1e6:.1f} MB  {len(d['y']):,} beats  OK")

In [ ]:
# --- B: build it instead (skip if you used A) --------------------------
# !python src/download.py
# !python src/build_dataset.py
# !python src/verify_dataset.py

## 4. The run — about 35 minutes

Three seeds × five folds. Each model trains on four folds of DS1 and is
explained on the fifth, which it never saw.

`-u` is load-bearing: without it Python buffers its output and this cell shows
nothing at all until the very end.

In [ ]:
!python -u src/stage5_explain.py --strategy class_weight --seeds 0 1 2

## 5. Look at the figures

In [ ]:
from IPython.display import Image, display

for f in ('attribution.png', 'faithfulness.png'):
    print(f)
    display(Image(f'artifacts/stage5/{f}'))

## 6. Package and download

In [ ]:
import os
import zipfile

from google.colab import files

NEED = ['stage5_results.json', 'attribution.png', 'faithfulness.png']
missing = [f for f in NEED if not os.path.exists(f'artifacts/stage5/{f}')]
assert not missing, (
    f'cell 4 did not finish — {missing} were never written.\\n'
    'Scroll up, read the end of cell 4, and send that text.')

OUT = '/content/stage5_results.zip'
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in NEED:
        z.write(f'artifacts/stage5/{f}', f)

print('packed:', NEED)
files.download(OUT)

## 7. What to send back, and what to read first

Send `stage5_results.zip` **and the full text output of cell 4**, as text rather
than a screenshot.

### Read these before you send

**The attribution table.** One row per heartbeat class, one column per region,
and a `*` marking the region the model leaned on most. The question the project
cares about: for class **S**, is `RR` the starred column? Class S is defined
clinically by a beat arriving early, so timing *should* dominate. If it does,
the attribution agrees with the ablation. If it does not, that is a genuine
finding and needs reporting, not hiding.

**The faithfulness lines.** Something like:

```
deletion  AUC  attributed 0.41   random 0.58   better in 15/15 runs
```

Attributed should be **lower** than random for deletion and **higher** for
insertion. If they come out level, the attribution is not informative and the
honest conclusion is that this method did not work on this model.

**Efficiency.** Parameter count around 146,000 and well under a millisecond per
beat. This is the embedded-feasibility claim.

### Stop and report, do not work around it

| Sign | What it means |
|---|---|
| `stage5_explain.py is not in the repository` | the script has not been merged yet |
| `folds.json is missing` | incomplete clone — pull main. **Do not run `make_folds.py`** |
| `AssertionError` about beat counts | wrong `ds1.npz` |
| every class starred on the same region | possible, but check the numbers are not all near zero first |
| all attributions ≈ 0 | the model is ignoring everything, which would mean something is wrong with how it was loaded |

There is no one-shot rule in this stage. Re-running costs nothing but time.